# 01 · Clustering desde cero: K-Means, jerárquico y DBSCAN

**Módulo 5 · Sesión 12** — Aprendizaje no supervisado

## Objetivos

Hasta aquí todos los modelos del curso tenían una $y$ que decía si la predicción era
buena. En este módulo no la hay: el algoritmo recibe solo $\mathbf{X}$ y debe encontrar
**estructura** — grupos, direcciones, anomalías — sin que nadie le diga cuál es la
correcta. Eso cambia dos cosas: qué significa "funciona", y cuánto depende el resultado
de los supuestos del algoritmo. Este notebook construye los tres algoritmos de clustering
de la sesión desde cero para ver exactamente qué supone cada uno:

1. Implementar **K-Means** (algoritmo de Lloyd) en ~20 líneas, ver que la inercia baja en
   cada paso, y verificar que coincide con `KMeans` de `scikit-learn`.
2. Medir cuánto depende el resultado de la **inicialización**, y qué arregla
   **K-Means++**.
3. Ver sobre datos construidos a propósito qué **supone** K-Means (grupos convexos,
   esféricos, de tamaño parecido) y qué pasa cuando no se cumple.
4. Implementar el clustering **jerárquico aglomerativo** y **DBSCAN** a mano, validarlos
   contra SciPy y `scikit-learn`, y ver qué resuelve cada uno que K-Means no.
5. Implementar el **método del codo** y el **coeficiente de silueta** a mano, y ver que
   ambos proponen un $k$ incluso cuando no hay grupos.

La teoría está en `01-clustering.md` y `02-validacion-clusters.md`.

**Paquetes:** `numpy`, `pandas`, `matplotlib`, `scipy`, `scikit-learn`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, fcluster, linkage
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos: cuatro grupos gaussianos

Datos sintéticos en 2D, para poder **ver** lo que hace cada algoritmo. Cuatro grupos
gaussianos con la misma dispersión — el caso ideal de K-Means. Guardamos las etiquetas
verdaderas `y` solo para medir al final; **ningún algoritmo las ve**.

In [ ]:
CENTROS = np.array([[-5, -5], [-5, 5], [5, -5], [5, 5]])
X, y = make_blobs(n_samples=600, centers=CENTROS, cluster_std=1.2, random_state=SEMILLA)


def graficar(X, etiquetas, eje, titulo, centroides=None):
    eje.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="tab10", s=10, vmin=0, vmax=9)
    if centroides is not None:
        eje.scatter(centroides[:, 0], centroides[:, 1], marker="X", s=180, c="black", edgecolor="white")
    eje.set_title(titulo)
    eje.set_xticks([])
    eje.set_yticks([])


fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
graficar(X, np.zeros(len(X), dtype=int), ejes[0], "Lo que ve el algoritmo")
graficar(X, y, ejes[1], "Los grupos verdaderos (que no ve)")
plt.show()

## 2. K-Means a mano: asignar y actualizar

K-Means busca $k$ centroides $\boldsymbol{\mu}_1, \dots, \boldsymbol{\mu}_k$ que
minimicen la **inercia**, la suma de distancias al cuadrado de cada punto a su centroide:

$$
J = \sum_{i=1}^{n} \min_{j} \, \lVert \mathbf{x}_i - \boldsymbol{\mu}_j \rVert^2
$$

Minimizar $J$ exactamente es NP-difícil. El algoritmo de **Lloyd** alterna dos pasos,
cada uno óptimo dado el otro:

1. **Asignación**: cada punto va al centroide más cercano (fija los centroides, minimiza
   $J$ sobre las asignaciones).
2. **Actualización**: cada centroide pasa a ser la media de sus puntos (fija las
   asignaciones, minimiza $J$ sobre los centroides — la media minimiza la suma de
   cuadrados, como en la S6).

Ninguno de los dos pasos puede subir $J$, así que la inercia baja o se queda igual en cada
iteración, y el algoritmo **converge** en un número finito de pasos — a un mínimo
**local**.

In [ ]:
def distancias_cuadrado(X, centroides):
    """Matriz n × k con ||x_i - mu_j||²."""
    return ((X[:, None, :] - centroides[None, :, :]) ** 2).sum(axis=2)


def kmeans_lloyd(X, centroides_iniciales, max_iter=100, guardar_pasos=False):
    centroides = centroides_iniciales.copy()
    historial, pasos = [], []
    for _ in range(max_iter):
        d2 = distancias_cuadrado(X, centroides)
        etiquetas = d2.argmin(axis=1)                       # paso 1: asignación
        historial.append(d2[np.arange(len(X)), etiquetas].sum())
        if guardar_pasos:
            pasos.append((etiquetas.copy(), centroides.copy()))
        nuevos = np.array([X[etiquetas == j].mean(axis=0) if np.any(etiquetas == j) else centroides[j]
                           for j in range(len(centroides))])  # paso 2: actualización
        if np.allclose(nuevos, centroides):
            break
        centroides = nuevos
    return etiquetas, centroides, np.array(historial), pasos


# Inicialización deliberadamente mala: los cuatro puntos más a la izquierda (mismo grupo).
inicio = X[np.argsort(X[:, 0])[:4]]
etiquetas, centroides, historial, pasos = kmeans_lloyd(X, inicio, guardar_pasos=True)

print(f"Iteraciones hasta converger: {len(historial)}")
print("Inercia por iteración:", np.round(historial, 1))
print("¿Baja siempre?", bool(np.all(np.diff(historial) <= 1e-9)))

fig, ejes = plt.subplots(1, 4, figsize=(16, 4))
for eje, i in zip(ejes, [0, 1, 2, len(pasos) - 1]):
    graficar(X, pasos[i][0], eje, f"Iteración {i} · J = {historial[i]:.0f}", pasos[i][1])
plt.show()

La inercia baja en **todas** las iteraciones —es una garantía del algoritmo, no una
casualidad— y se estanca en pocas. Validamos contra `scikit-learn` partiendo de los mismos
centroides (`n_init=1` para que no reinicie):

In [ ]:
km = KMeans(n_clusters=4, init=inicio, n_init=1, max_iter=100, tol=0).fit(X)
print(f"Inercia a mano:         {historial[-1]:.4f}")
print(f"Inercia de scikit-learn: {km.inertia_:.4f}")
print(f"ARI entre las dos particiones: {adjusted_rand_score(etiquetas, km.labels_):.4f}")
print(f"Centroides coinciden: {np.allclose(np.sort(centroides, axis=0), np.sort(km.cluster_centers_, axis=0))}")

## 3. Cuánto depende de la inicialización, y qué arregla K-Means++

Lloyd converge a un mínimo local, y **cuál** depende de dónde empiece. Repetimos 200 veces
la inicialización más simple —$k$ puntos al azar— y miramos la distribución de la inercia
final.

In [ ]:
def inicio_aleatorio(X, k, rng):
    return X[rng.choice(len(X), k, replace=False)]


def inicio_kmeanspp(X, k, rng):
    """K-Means++: el primer centroide al azar; cada siguiente, con probabilidad ∝ D(x)²."""
    centroides = [X[rng.integers(len(X))]]
    for _ in range(1, k):
        d2 = distancias_cuadrado(X, np.array(centroides)).min(axis=1)
        centroides.append(X[rng.choice(len(X), p=d2 / d2.sum())])
    return np.array(centroides)


N_REP = 200
resultados = {}
for nombre, init in [("aleatoria", inicio_aleatorio), ("K-Means++", inicio_kmeanspp)]:
    inercias = []
    for _ in range(N_REP):
        _, _, hist, _ = kmeans_lloyd(X, init(X, 4, rng))
        inercias.append(hist[-1])
    resultados[nombre] = np.array(inercias)

optimo = min(r.min() for r in resultados.values())
tabla = pd.DataFrame({
    "inercia mínima": {n: r.min() for n, r in resultados.items()},
    "inercia mediana": {n: np.median(r) for n, r in resultados.items()},
    "inercia máxima": {n: r.max() for n, r in resultados.items()},
    "% corridas en el óptimo": {n: 100 * np.mean(r < optimo * 1.001) for n, r in resultados.items()},
})
print(tabla.round(1).to_string())

fig, eje = plt.subplots(figsize=(7, 4))
for nombre, r in resultados.items():
    eje.hist(r, bins=40, alpha=0.6, label=nombre)
eje.set_xlabel("Inercia final (200 inicializaciones)")
eje.set_ylabel("Corridas")
eje.legend()
eje.set_title("La inicialización decide en qué mínimo local se cae")
plt.show()

Con inicialización aleatoria, el 30 % de las corridas termina en un mínimo local
claramente peor: dos centroides se reparten un mismo grupo mientras otro
centroide cubre dos grupos a la vez. **K-Means++** (Arthur y Vassilvitskii, 2007) elige cada
centroide inicial con probabilidad proporcional a la distancia al cuadrado a los ya
elegidos, así que tiende a empezar con un centroide por grupo. No garantiza el óptimo,
pero casi siempre lo alcanza — y es lo que `scikit-learn` usa por defecto, además de
repetir la corrida `n_init` veces y quedarse con la de menor inercia. Ese es el ajuste
práctico: **K-Means++ y varias inicializaciones**, no una sola corrida.

In [ ]:
peor = resultados["aleatoria"].argmax()
rng_peor = np.random.default_rng(SEMILLA)
for _ in range(peor + 1):
    inicio_malo = inicio_aleatorio(X, 4, rng_peor)
etiq_malo, cent_malo, hist_malo, _ = kmeans_lloyd(X, inicio_malo)
etiq_pp, cent_pp, hist_pp, _ = kmeans_lloyd(X, inicio_kmeanspp(X, 4, np.random.default_rng(SEMILLA)))

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
graficar(X, etiq_malo, ejes[0], f"Mínimo local · J = {hist_malo[-1]:.0f}", cent_malo)
graficar(X, etiq_pp, ejes[1], f"K-Means++ · J = {hist_pp[-1]:.0f}", cent_pp)
plt.show()

## 4. Qué supone K-Means, y qué pasa cuando no se cumple

La asignación al centroide más cercano parte el plano en **celdas de Voronoi**: regiones
convexas separadas por rectas (hiperplanos en más dimensiones). Y la inercia, por ser una
suma de cuadrados, favorece grupos **esféricos** y de **tamaño parecido**: un grupo grande
y disperso aporta más inercia que uno pequeño, y K-Means lo corta para repartirla.

Tres datasets construidos para violar cada supuesto, más el caso ideal. K-Means recibe el
$k$ verdadero en todos; medimos con el **índice de Rand ajustado** (ARI) contra los grupos
verdaderos (1 = partición idéntica; 0 = lo que daría el azar).

In [ ]:
X_iso, y_iso = make_blobs(n_samples=600, centers=3, cluster_std=1.0, random_state=SEMILLA)

# Tres grupos alargados en diagonal, uno junto a otro: la dirección larga no es la que los separa.
X_ani, y_ani = [], []
for j, cx in enumerate([0, 3.5, 7]):
    largo, corto = rng.normal(0, 2.5, 200), rng.normal(0, 0.35, 200)
    X_ani.append(np.c_[cx + (largo - corto) / np.sqrt(2), (largo + corto) / np.sqrt(2)])
    y_ani += [j] * 200
X_ani, y_ani = np.vstack(X_ani), np.array(y_ani)

X_var, y_var = make_blobs(n_samples=600, centers=[[6, 0], [0, 0], [-6, 0]],
                          cluster_std=[0.5, 3.0, 0.5], random_state=SEMILLA)             # varianzas distintas
X_tam, y_tam = make_blobs(n_samples=[500, 50, 50], centers=[[0, 0], [5, 4], [-5, 4]],
                          cluster_std=[2.0, 0.5, 0.5], random_state=SEMILLA)             # tamaños desiguales
X_lun, y_lun = make_moons(n_samples=600, noise=0.07, random_state=SEMILLA)

casos = [("Ideal: esféricos, mismo tamaño", X_iso, y_iso, 3),
         ("Alargados (anisotrópicos)", X_ani, y_ani, 3),
         ("Varianzas distintas", X_var, y_var, 3),
         ("Tamaños desiguales", X_tam, y_tam, 3),
         ("No convexos (medias lunas)", X_lun, y_lun, 2)]

fig, ejes = plt.subplots(2, 5, figsize=(20, 7.5))
ari_kmeans = {}
for col, (nombre, Xc, yc, k) in enumerate(casos):
    kmc = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(Xc)
    ari_kmeans[nombre] = adjusted_rand_score(yc, kmc.labels_)
    graficar(Xc, yc, ejes[0, col], nombre)
    graficar(Xc, kmc.labels_, ejes[1, col], f"K-Means (k={k}) · ARI = {ari_kmeans[nombre]:.2f}", kmc.cluster_centers_)
ejes[0, 0].set_ylabel("Grupos verdaderos", fontsize=12)
ejes[1, 0].set_ylabel("K-Means", fontsize=12)
plt.show()

K-Means no "se equivoca" en los cuatro casos de la derecha: **minimiza la inercia
correctamente**, y la partición de mínima inercia no es la que un humano dibujaría. Cuando
los grupos son alargados, corta a lo ancho; cuando un grupo es mucho más disperso, le roba
puntos a los vecinos compactos; cuando uno es mucho más grande, lo parte; y las medias
lunas no son separables por rectas. Todo esto se ve en 2D. En 11 dimensiones —Wine
Quality, notebook 02— no se ve, y hay que **sospecharlo** a partir de los diagnósticos.

## 5. Clustering jerárquico aglomerativo

Otra idea: empezar con $n$ grupos de un punto y **fusionar** en cada paso los dos grupos
más cercanos, hasta que quede uno. El resultado no es una partición sino una jerarquía
completa —el **dendrograma**—, que se corta a la altura que dé el número de grupos
deseado. Lo único que hay que definir es la distancia **entre grupos** (el *linkage*):

| Linkage | Distancia entre grupos $A$ y $B$ | Tiende a |
|---|---|---|
| *single* | mínima entre un punto de $A$ y uno de $B$ | encadenar: sigue formas alargadas, sensible a puentes de ruido |
| *complete* | máxima entre un punto de $A$ y uno de $B$ | grupos compactos de diámetro parecido |
| *average* | media de todas las distancias entre pares | un compromiso |
| *Ward* | aumento de inercia al fusionar | grupos esféricos de tamaño parecido (el K-Means jerárquico) |

Lo implementamos de la forma más directa —una matriz de distancias entre grupos que se
recalcula tras cada fusión, $O(n^3)$— sobre 40 puntos, y comparamos las alturas de fusión
con `scipy.cluster.hierarchy.linkage`.

In [ ]:
def aglomerativo(X, metodo="single"):
    """Devuelve la lista de fusiones (i, j, distancia) en orden; grupos indexados como SciPy."""
    n = len(X)
    grupos = {i: [i] for i in range(n)}
    D = np.sqrt(distancias_cuadrado(X, X))
    agregador = {"single": np.min, "complete": np.max, "average": np.mean}[metodo]
    fusiones = []
    siguiente = n
    while len(grupos) > 1:
        ids = list(grupos)
        mejor = (np.inf, None, None)
        for a in range(len(ids)):
            for b in range(a + 1, len(ids)):
                d = agregador(D[np.ix_(grupos[ids[a]], grupos[ids[b]])])
                if d < mejor[0]:
                    mejor = (d, ids[a], ids[b])
        d, i, j = mejor
        grupos[siguiente] = grupos.pop(i) + grupos.pop(j)
        fusiones.append((min(i, j), max(i, j), d, len(grupos[siguiente])))
        siguiente += 1
    return np.array(fusiones)


X_peq = X[rng.choice(len(X), 40, replace=False)]
for metodo in ["single", "complete", "average"]:
    Z_mano = aglomerativo(X_peq, metodo)
    Z_scipy = linkage(X_peq, method=metodo)
    print(f"{metodo:<9} alturas de fusión iguales a SciPy: {np.allclose(Z_mano[:, 2], Z_scipy[:, 2])}   "
          f"(última fusión a distancia {Z_mano[-1, 2]:.2f})")

Las 39 alturas de fusión coinciden con SciPy para los tres *linkages*. El dendrograma
dibuja esas fusiones: cada unión a la altura de la distancia a la que ocurrió. Un salto
grande entre dos alturas consecutivas sugiere que ahí hay un corte natural.

In [ ]:
Z = linkage(X_peq, method="average")
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
dendrogram(Z, ax=ejes[0], color_threshold=Z[-3, 2], no_labels=True)
ejes[0].axhline(Z[-3, 2], color="gray", ls="--")
ejes[0].set_title("Dendrograma (average) · corte en 4 grupos")
ejes[0].set_ylabel("Distancia de fusión")
etiq_jer = fcluster(Z, t=4, criterion="maxclust")
graficar(X_peq, etiq_jer, ejes[1], "Los 4 grupos del corte")
plt.show()

saltos = np.diff(Z[:, 2])
print("Últimas alturas de fusión:", np.round(Z[-6:, 2], 2))
print(f"El mayor salto está entre la fusión {np.argmax(saltos) + 1} y la {np.argmax(saltos) + 2} "
      f"→ quedan {len(X_peq) - np.argmax(saltos) - 1} grupos")

### Lo que cada linkage ve en las medias lunas

El *linkage* no es un detalle: decide qué forma de grupo puede encontrar el algoritmo.
Sobre las medias lunas, que K-Means no pudo separar:

In [ ]:
fig, ejes = plt.subplots(1, 4, figsize=(16, 3.8))
for eje, metodo in zip(ejes, ["single", "complete", "average", "ward"]):
    etiq = fcluster(linkage(X_lun, method=metodo), t=2, criterion="maxclust")
    graficar(X_lun, etiq, eje, f"{metodo} · ARI = {adjusted_rand_score(y_lun, etiq):.2f}")
plt.show()

*Single linkage* separa las dos lunas perfectamente porque solo necesita que cada luna sea
una **cadena** de puntos cercanos; los otros tres cortan a lo ancho, como K-Means. La
contrapartida es la misma cualidad: basta un puente de unos pocos puntos de ruido entre dos
grupos para que *single* los una en uno solo. El jerárquico cuesta $O(n^2)$ de memoria (la
matriz de distancias) y no escala a cientos de miles de filas; a cambio, no exige fijar $k$
antes de mirar.

## 6. DBSCAN: grupos como regiones densas

Un tercer punto de vista: un grupo es una **región densa** separada de otras por regiones
poco densas. DBSCAN (Ester et al., 1996) tiene dos parámetros, un radio `eps` y un mínimo
de puntos `min_samples`, y clasifica cada punto como:

- **núcleo** (*core*): tiene al menos `min_samples` puntos (contándose) a distancia
  $\leq$ `eps`;
- **borde**: no es núcleo, pero está a distancia $\leq$ `eps` de un núcleo;
- **ruido**: ninguna de las dos.

Los grupos son las componentes conexas de los puntos núcleo (dos núcleos a distancia
$\leq$ `eps` están en el mismo grupo); los bordes se pegan al grupo de algún núcleo vecino;
el ruido queda fuera con etiqueta $-1$. No hay que fijar $k$, los grupos pueden tener
cualquier forma, y **hay puntos que no pertenecen a ningún grupo** — es también un detector
de anomalías.

In [ ]:
def dbscan_mano(X, eps, min_samples):
    n = len(X)
    D = np.sqrt(distancias_cuadrado(X, X))
    vecinos = [np.flatnonzero(D[i] <= eps) for i in range(n)]
    nucleo = np.array([len(v) >= min_samples for v in vecinos])
    etiquetas = np.full(n, -1)
    grupo = 0
    for i in range(n):
        if not nucleo[i] or etiquetas[i] != -1:
            continue
        etiquetas[i] = grupo
        pendientes = list(vecinos[i])
        while pendientes:                      # expansión del grupo desde el núcleo i
            j = pendientes.pop()
            if etiquetas[j] == -1:
                etiquetas[j] = grupo           # borde o núcleo aún sin grupo
                if nucleo[j]:
                    pendientes.extend(vecinos[j])
        grupo += 1
    return etiquetas, nucleo


EPS = 0.15
etiq_mano, nucleo = dbscan_mano(X_lun, eps=EPS, min_samples=5)
etiq_sk = DBSCAN(eps=EPS, min_samples=5).fit(X_lun).labels_
print(f"Grupos a mano: {etiq_mano.max() + 1}, ruido: {(etiq_mano == -1).sum()}   |   "
      f"scikit-learn: {etiq_sk.max() + 1}, ruido: {(etiq_sk == -1).sum()}")
print(f"ARI entre las dos: {adjusted_rand_score(etiq_mano, etiq_sk):.4f}   "
      f"núcleos coinciden: {np.array_equal(np.flatnonzero(nucleo), DBSCAN(eps=EPS, min_samples=5).fit(X_lun).core_sample_indices_)}")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
graficar(X_lun, etiq_mano, ejes[0], f"DBSCAN (eps={EPS}) · ARI = {adjusted_rand_score(y_lun, etiq_mano):.2f}")
tipo = np.where(etiq_mano == -1, 0, np.where(nucleo, 2, 1))
ejes[1].scatter(X_lun[:, 0], X_lun[:, 1], c=tipo, cmap="viridis", s=10)
ejes[1].set_title("Amarillo: núcleo · verde: borde · morado: ruido")
ejes[1].set_xticks([])
ejes[1].set_yticks([])
plt.show()

### `eps` decide todo

DBSCAN no pide $k$, pero `eps` es igual de decisivo, y **no** hay un valor razonable a
priori: depende de la escala de los datos (por eso, como K-Means, exige estandarizar). Un
barrido sobre las medias lunas:

In [ ]:
filas = []
for eps in [0.05, 0.1, 0.15, 0.2, 0.3, 0.5]:
    etiq = DBSCAN(eps=eps, min_samples=5).fit(X_lun).labels_
    filas.append({"eps": eps, "grupos": etiq.max() + 1, "% ruido": 100 * np.mean(etiq == -1),
                  "ARI": adjusted_rand_score(y_lun, etiq)})
print(pd.DataFrame(filas).round(2).to_string(index=False))

Con `eps` pequeño, casi todo es ruido y los grupos se fragmentan; con `eps` grande, las
dos lunas se tocan y se funden en una. Hay una ventana intermedia donde funciona. La
heurística estándar para encontrarla (Ester et al.) es el **gráfico de k-distancias**:
para cada punto, la distancia a su `min_samples`-ésimo vecino, ordenada de mayor a menor.
Los puntos de las regiones densas tienen esa distancia pequeña y parecida; el ruido, grande.
El "codo" de la curva es un candidato para `eps`.

In [ ]:
MIN_SAMPLES = 5
d_k = NearestNeighbors(n_neighbors=MIN_SAMPLES).fit(X_lun).kneighbors(X_lun)[0][:, -1]
d_k = np.sort(d_k)[::-1]

fig, eje = plt.subplots(figsize=(7, 4))
eje.plot(d_k)
eje.axhline(0.15, color="gray", ls="--", label="eps ≈ 0.15")
eje.set_xlabel("Puntos, ordenados por distancia a su 5.º vecino")
eje.set_ylabel("Distancia al 5.º vecino más cercano")
eje.set_title("Gráfico de k-distancias: el codo sugiere eps")
eje.legend()
plt.show()

## 7. Elegir $k$: codo y silueta, a mano

K-Means y el jerárquico necesitan un $k$; DBSCAN necesita un `eps`. En 2D se elige mirando;
en 11 dimensiones hacen falta números. Los dos más usados:

- **Método del codo**: la inercia $J(k)$ siempre baja al subir $k$ (con $k=n$ vale cero),
  pero si hay $k^*$ grupos reales, baja mucho hasta $k^*$ y poco después. Se busca el
  "codo".
- **Coeficiente de silueta** (Rousseeuw, 1987): para cada punto $i$, con $a_i$ su distancia
  media a los puntos de **su** grupo y $b_i$ la distancia media al grupo **vecino** más
  cercano,

$$
s_i = \frac{b_i - a_i}{\max(a_i, b_i)} \in [-1, 1]
$$

vale cerca de 1 si el punto está mucho más cerca de su grupo que del vecino, cerca de 0
si está en la frontera, y negativo si probablemente está mal asignado. La silueta global
es el promedio de los $s_i$, y se elige el $k$ que la maximiza.

In [ ]:
def silueta_mano(X, etiquetas):
    D = np.sqrt(distancias_cuadrado(X, X))
    grupos = np.unique(etiquetas)
    s = np.zeros(len(X))
    for i in range(len(X)):
        propio = etiquetas == etiquetas[i]
        if propio.sum() == 1:
            continue                                        # convención: s = 0 para singletons
        a = D[i, propio].sum() / (propio.sum() - 1)         # excluye la distancia a sí mismo
        b = min(D[i, etiquetas == g].mean() for g in grupos if g != etiquetas[i])
        s[i] = (b - a) / max(a, b)
    return s


ks = range(2, 11)
inercias, siluetas, siluetas_sk = [], [], []
for k in ks:
    kmc = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X)
    inercias.append(kmc.inertia_)
    siluetas.append(silueta_mano(X, kmc.labels_).mean())
    siluetas_sk.append(silhouette_score(X, kmc.labels_))
print(f"Silueta a mano coincide con scikit-learn: {np.allclose(siluetas, siluetas_sk)}")
print(pd.DataFrame({"k": ks, "inercia": np.round(inercias), "silueta": np.round(siluetas, 3)}).to_string(index=False))

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(ks, inercias, "o-")
ejes[0].set_xlabel("k")
ejes[0].set_ylabel("Inercia")
ejes[0].set_title("Método del codo")
ejes[1].plot(ks, siluetas, "o-")
ejes[1].set_xlabel("k")
ejes[1].set_ylabel("Silueta media")
ejes[1].set_title("Coeficiente de silueta")
plt.show()

Los dos coinciden en $k = 4$, que es el verdadero: la inercia se quiebra ahí y la silueta
tiene su máximo. La silueta es más cómoda porque da un máximo, no un "codo" que hay que
juzgar a ojo — pero mide **compacidad y separación** con distancias euclídeas, así que
hereda los supuestos de K-Means: sobre las medias lunas, la partición **correcta** de
DBSCAN (ARI 1.0) tiene silueta −0.02, y la incorrecta de K-Means (ARI 0.24), 0.49.

In [ ]:
etiq_db = DBSCAN(eps=0.15, min_samples=5).fit(X_lun).labels_
etiq_km = KMeans(n_clusters=2, n_init=10, random_state=SEMILLA).fit(X_lun).labels_
print(f"Medias lunas · silueta de DBSCAN (ARI 1.00): {silhouette_score(X_lun, etiq_db):.3f}   "
      f"silueta de K-Means (ARI {adjusted_rand_score(y_lun, etiq_km):.2f}): {silhouette_score(X_lun, etiq_km):.3f}")

### La trampa: siempre hay un "mejor $k$"

Lo más importante de la sección. Ni el codo ni la silueta comprueban que **exista**
estructura de grupos: comparan particiones entre sí, y una de ellas siempre gana. Sobre
puntos **uniformes** en un cuadrado —donde no hay grupos por construcción— ambos
criterios proponen un $k$ con la misma naturalidad:

In [ ]:
X_uni = rng.uniform(-5, 5, size=(600, 2))
inercias_u, siluetas_u = [], []
for k in ks:
    kmc = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA).fit(X_uni)
    inercias_u.append(kmc.inertia_)
    siluetas_u.append(silhouette_score(X_uni, kmc.labels_))

fig, ejes = plt.subplots(1, 3, figsize=(16, 4))
graficar(X_uni, KMeans(n_clusters=ks[int(np.argmax(siluetas_u))], n_init=10, random_state=SEMILLA).fit(X_uni).labels_,
         ejes[0], f"Uniforme, K-Means con el 'mejor' k = {ks[int(np.argmax(siluetas_u))]}")
ejes[1].plot(ks, inercias_u, "o-")
ejes[1].set_title("Inercia: también baja, también con 'codo'")
ejes[1].set_xlabel("k")
ejes[2].plot(ks, siluetas_u, "o-", label="uniforme (sin grupos)")
ejes[2].plot(ks, siluetas, "o-", label="4 grupos reales")
ejes[2].set_title(f"Silueta máxima: {max(siluetas_u):.2f} sin grupos, {max(siluetas):.2f} con ellos")
ejes[2].set_xlabel("k")
ejes[2].legend()
plt.show()
print(f"Silueta máxima sin grupos: {max(siluetas_u):.3f} (k = {ks[int(np.argmax(siluetas_u))]})   "
      f"con 4 grupos reales: {max(siluetas):.3f} (k = 4)")

La silueta máxima de los datos uniformes es 0.41 — no cero. El valor **absoluto** de la
silueta, y no solo cuál $k$ la maximiza, es lo que informa: por debajo de ≈0.25 la
"estructura" encontrada no se distingue de lo que K-Means impone a datos sin grupos
(Kaufman y Rousseeuw, 1990, dan 0.25 como umbral orientativo; 0.5 como "estructura
razonable"; 0.7 como "estructura fuerte").
El notebook 02 hace esta comparación de forma más rigurosa sobre Wine Quality, con una
**referencia nula** construida a partir de los propios datos.

## Resumen

| Pregunta | Respuesta medida |
|---|---|
| ¿Qué hace K-Means? | Alterna asignación y media; la inercia baja en cada paso; converge a un mínimo local. Coincide con `KMeans` a $10^{-4}$ |
| ¿Importa la inicialización? | Sí: con $k$ puntos al azar, el 30 % de las corridas cae en un mínimo local peor; con K-Means++, el 10 %. `n_init` remata |
| ¿Qué supone K-Means? | Grupos convexos, esféricos, de tamaño y varianza parecidos. Alargados, dispares o no convexos: ARI cae de 1.0 a 0.24–0.73 sin ninguna advertencia |
| ¿Qué añade el jerárquico? | Una jerarquía completa sin fijar $k$; el *linkage* decide la forma: *single* separa las lunas (ARI 1.0), los demás no |
| ¿Y DBSCAN? | Grupos de cualquier forma, ruido explícito; `eps` decide todo y el gráfico de k-distancias ayuda a elegirlo |
| ¿Cómo elegir $k$? | Codo y silueta coinciden en el $k$ real cuando hay grupos (silueta 0.77)… y proponen un $k$ igual de convencidos cuando no los hay (silueta 0.41 sobre puntos uniformes) |